In [1]:
import json
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

all_rows = []

for path in sorted(DATA_DIR.glob("*/*.json")):
    with open(path, encoding="utf-8") as f:
        all_rows.extend(json.load(f))

df = pd.DataFrame(all_rows)

print("Total records:", len(df))
print("Columns:", len(df.columns))

display(df.groupby("store").size())

Total records: 11680
Columns: 15


store
barbora    5861
rimi       5819
dtype: int64

In [2]:
df_clean = df.drop_duplicates(
    subset=["store", "id"]
).copy()

print("Duplicates removed:", len(df) - len(df_clean))
print("Remaining products:", len(df_clean))

Duplicates removed: 26
Remaining products: 11654


In [3]:
rules = {
    "invalid_price": (
        df_clean["price"].isna()
        | (df_clean["price"] <= 0)
    ),

    "invalid_regular_price": (
        df_clean["regular_price"].isna()
        | (df_clean["regular_price"] <= 0)
    ),

    "invalid_unit_price": (
        df_clean["unit_price"].isna()
        | (df_clean["unit_price"] <= 0)
    ),

    "reversed_prices": (
        df_clean["price"] > df_clean["regular_price"]
    ),

    "inconsistent_promotion": (
        df_clean["on_promo"]
        & (df_clean["price"] >= df_clean["regular_price"])
    ),

    "loyalty_without_discount": (
        df_clean["loyalty_card_required"]
        & ~df_clean["on_promo"]
    ),

    "invalid_unit": (
        ~df_clean["unit"].isin(["kg", "l", "pcs"])
    ),
}

quality_flags = pd.DataFrame(rules)

display(quality_flags.sum().to_frame("Affected products"))

exclude_mask = quality_flags.any(axis=1)

df_excluded = df_clean[exclude_mask].copy()
df_clean = df_clean[~exclude_mask].copy()

print("Excluded products:", len(df_excluded))
print("Remaining products:", len(df_clean))

,Affected products
invalid_price,1
invalid_regular_price,1
invalid_unit_price,485
reversed_prices,30
inconsistent_promotion,52
loyalty_without_discount,30
invalid_unit,484


Excluded products: 568
Remaining products: 11086


In [4]:
category_mapping = {
    # Barbora
    "darzoves-ir-vaisiai": "produce",
    "pieno-gaminiai-kiausiniai-ir-majonezas": "dairy-eggs",
    "duonos-gaminiai-ir-konditerija": "bread",
    "mesa-zuvis-ir-kulinarija": "meat-and-fish",

    # Rimi
    "vaisiai-darzoves-ir-geles": "produce",
    "pieno-produktai-ir-kiausiniai": "dairy-eggs",
    "duonos-gaminiai": "bread",
    "mesa-ir-zuvis": "meat-and-fish",
    "augaliniai-produktai": "plant-based",

    # Shared
    "bakaleja": "pantry",
    "saldytas-maistas": "frozen",
}

df_clean["main_category"] = (
    df_clean["category"]
    .str.split("/")
    .str[0]
    .str.strip("-")
    .map(category_mapping)
    .fillna("other")
)

display(
    df_clean.groupby(["main_category", "store"])
    .size()
    .unstack(fill_value=0)
)

store,barbora,rimi
main_category,,
bread,570,245
dairy-eggs,1161,863
frozen,602,468
meat-and-fish,658,826
other,4,0
pantry,2476,2502
plant-based,0,101
produce,313,297


In [5]:
df_clean = df_clean.reset_index(drop=True)

final_summary = df_clean.groupby("store").agg(
    products=("id", "size"),
    promotions=("on_promo", "sum"),
    loyalty_products=("loyalty_card_required", "sum"),
    missing_regular_unit_price=(
        "regular_unit_price",
        lambda x: x.isna().sum()
    )
)

display(final_summary)

print("Original records:", len(df))
print("Final records:", len(df_clean))
print("Total removed:", len(df) - len(df_clean))

print(
    "Data retained:",
    f"{len(df_clean) / len(df):.2%}"
)

,products,promotions,loyalty_products,missing_regular_unit_price
store,,,,
barbora,5784,226,226,226
rimi,5302,1631,1293,338


Original records: 11680
Final records: 11086
Total removed: 594
Data retained: 94.91%


In [6]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df_clean.to_parquet(
    PROCESSED_DIR / "products.parquet",
    index=False
)

df_excluded.to_parquet(
    PROCESSED_DIR / "excluded_products.parquet",
    index=False
)

print("Processed datasets saved.")

Processed datasets saved.
